# Wanderings through duckdb and polars

Findings: 
1. gosh this duckdb and polars stuff is fast.
2. now we get to figure out the useful views we need and we can work on the fast because parquet has modern bindings in so many places, from R to python.

In [1]:
import duckdb 
import polars as pl
from pathlib import Path



In [2]:
# Various paths to data sets in the repo

NATIONAL_CSB_PARQUET_PATH = Path("../out/national1825.parquet")
CSB_LAND_COVER_CODES_PARQUET_PATH = Path("../useful-views/land_cover_codes.parquet")

In [3]:
national = None
national = duckdb.connect()
national.execute("LOAD spatial")

In [19]:
# Excluding `Shape` as that is a multipolygon column.
# It is not easily rendered and not needed for this example.

"""
Also note, displaying this on your local should be pretty easy, but displaying on github through the  
html render of the notebook may require that your browser is gigantically wide, or text super small. 
On my 27 inch display its readable, but I for sure have my glasses on :) 

"""

national.sql(
    """
    SELECT * EXCLUDE Shape
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).show()



┌──────────┬─────────────────┬──────────┬────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬───────────┬──────────┬─────────┬─────────┬──────────┬───────────────────┬──────────────┬──────────┬────────────────────┬────────────────────┐
│ OBJECTID │      CSBID      │ CSBYEARS │      CSBACRES      │ CDL2018 │ CDL2019 │ CDL2020 │ CDL2021 │ CDL2022 │ CDL2023 │ CDL2024 │ CDL2025 │ STATEFIPS │ STATEASD │   ASD   │  CNTY   │ CNTYFIPS │     INSIDE_X      │   INSIDE_Y   │ ORIG_FID │    Shape_Length    │     Shape_Area     │
│  int64   │     varchar     │ varchar  │       double       │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  int32  │  varchar  │ varchar  │ varchar │ varchar │ varchar  │      double       │    double    │  int32   │       double       │       double       │
├──────────┼─────────────────┼──────────┼────────────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼───

In [17]:

# if controlled out of the notebook flow - uncomment to see the nature of the multipolygon.

if 0:
    national.sql(
        """
        SELECT Shape
        FROM read_parquet($path)
        LIMIT 20
        """,
        params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
    ).show()

In [14]:
shape_sample = None
shape_sample = national.sql(
    """
    SELECT ST_AsText(Shape) AS Shape
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(NATIONAL_CSB_PARQUET_PATH)},
).pl().slice(17, 1)

#just to give the reader a sense of the specific shape at this position...



In [12]:
print(shape_sample)

shape: (2, 1)
┌─────────────────────────────────┐
│ Shape                           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ MULTIPOLYGON (((-663619.003499… │
│ MULTIPOLYGON (((-664168.977399… │
└─────────────────────────────────┘


In [28]:
csb_codes = None
csb_codes = duckdb.connect()
csb_codes.execute("LOAD spatial")

In [31]:
csb_codes.sql(
    """
    SELECT *
    FROM read_parquet($path)
    LIMIT 20
    """,
    params={"path": str(CSB_LAND_COVER_CODES_PARQUET_PATH)},
)

┌─────────────────┬──────────────────────────┐
│ land_cover_code │     land_cover_name      │
│      int64      │         varchar          │
├─────────────────┼──────────────────────────┤
│               0 │ Background               │
│               1 │ Corn                     │
│               2 │ Cotton                   │
│               3 │ Rice                     │
│               4 │ Sorghum                  │
│               5 │ Soybeans                 │
│               6 │ Sunflower                │
│              10 │ Peanuts                  │
│              11 │ Tobacco                  │
│              12 │ Sweet Corn               │
│              13 │ Pop or Orn Corn          │
│              14 │ Mint                     │
│              21 │ Barley                   │
│              22 │ Durum Wheat              │
│              23 │ Spring Wheat             │
│              24 │ Winter Wheat             │
│              25 │ Other Small Grains       │
│            

In [29]:
# tmp.write_csv("../out/tmp.csv")

# this makes a nearly useless csv dont do this.

In [21]:
pl.build_info()

{'version': '1.44.2'}

In [23]:
pl.thread_pool_size()

8